# Operasyonel Yorgunluk ve Kırılma Noktası (Fatigue Analysis)

**Analiz Konusu ve Hedef:**
Günlük sefer sayısı ve toplam operasyon süresinin (dk) otobüslerin dayanıklılığı üzerindeki etkisinin bulunması.

**Beklenen Çıktı ve Aksiyon:**
Bir aracın gün içinde belli bir sefer sayısını veya çalışma saatini aştığında arıza riskinin logaritmik arttığı 'Kırılma Noktası' eşiğini (Threshold) belirlemek.

In [1]:
# BOLUM 1: Veri Yukleme + Sefer Yogunlugu Hazirligi
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Ariza verisi
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv', low_memory=False)
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
print(f'Ariza: {len(df):,} kayit, {df["KAPINO"].nunique():,} arac')
print(f'Tarih: {df["OLAYTARIHI"].min()} -> {df["OLAYTARIHI"].max()}')

# Arac gunluk hatlar (sefer)
sefer = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv', low_memory=False)
sefer['TARIH'] = pd.to_datetime(sefer['TARIH'], format='mixed')
SEFER_KOL = 'SEFER_SAYISI'
print(f'Sefer kayitlari: {len(sefer):,}, {sefer["KAPINO"].nunique():,} arac')

# Arac × Gun bazinda sefer toplami (ayni arac ayni gun coklu hatta)
arac_gun_sefer = sefer.groupby(['KAPINO','TARIH'])[SEFER_KOL].sum().reset_index()
arac_gun_sefer.columns = ['KAPINO','TARIH','GUNLUK_SEFER']

# Arac bazinda temel istatistikler
arac_istat = arac_gun_sefer.groupby('KAPINO').agg(
    sefer_gun_sayisi=('TARIH','nunique'),
    sefer_ort=('GUNLUK_SEFER','mean'),
    sefer_std=('GUNLUK_SEFER','std'),
    sefer_top=('GUNLUK_SEFER','sum'),
    sefer_max=('GUNLUK_SEFER','max'),
).reset_index()
print(f'\nArac sefer istatistikleri: {len(arac_istat):,} arac')
print(arac_istat[['sefer_ort','sefer_std','sefer_top']].describe().round(2))

# Arac meta (yas, garaj)
arac_meta = df.groupby('KAPINO').agg(
    GARAJ=('GARAJ','first'),
    MARKA=('MARKA','first'),
    MODELYILI=('MODELYILI','first'),
    ARACCINSI=('ARACCINSI','first'),
).reset_index()
arac_meta['yas'] = 2025 - arac_meta['MODELYILI']

# Ariza tarafindaki GUNLUK_SEFER_SAYISI kontrolu
print(f'\nAriza icindeki GUNLUK_SEFER_SAYISI istatistikleri:')
print(df['GUNLUK_SEFER_SAYISI'].describe().round(2))
print(f'NaN: {df["GUNLUK_SEFER_SAYISI"].isna().sum():,}')


Ariza: 58,559 kayit, 3,509 arac
Tarih: 2025-01-01 00:30:56 -> 2025-06-30 23:29:56
Sefer kayitlari: 1,320,647, 6,761 arac

Arac sefer istatistikleri: 6,761 arac
       sefer_ort  sefer_std  sefer_top
count    6761.00    6738.00    6761.00
mean        9.45       3.29    1441.93
std         2.15       1.56     413.37
min         1.00       0.00       1.00
25%         8.19       2.15    1217.00
50%         9.34       3.29    1451.00
75%        10.42       4.10    1668.00
max        24.84      12.18    3720.00

Ariza icindeki GUNLUK_SEFER_SAYISI istatistikleri:
count    58559.00
mean         8.83
std          4.22
min          0.00
25%          6.00
50%          8.00
75%         11.00
max         40.00
Name: GUNLUK_SEFER_SAYISI, dtype: float64
NaN: 0


---

## 🔄 TAMAMLAYICI BAKIŞ AÇISI

# ⏳ ANALİZ 6: OPERASYONEL YORGUNLUK VE KIRILMA EŞİĞİ

### 📌 Analiz Amacı
Araçların günlük çalışma yoğunluğunun (sefer sayısı ve çalışma saati) arıza olasılığını nasıl tetiklediğini bulmak.

### 🛠️ Yapılacak İşlemler
1. **Yoğunluk Hesaplama:** Günlük sefer sayısı vs. Arıza gerçekleşme saati.
2. **Kırılma Noktası Analizi:** Hangi saatten veya seferden sonra arıza riski %50'nin üzerine çıkıyor?

### 🎯 Beklenen Çıktı
- **Operasyonel Limitler:** Örn: "Mercedes Citaro araçlar için günlük 16 saat çalışma sınırı konulmalıdır."

---
## 2. Sefer Sayisi Dagilimi + Kirilma Noktasi Tespiti

Soru: Gunluk sefer sayisinin arizayla iliskisi nasil? Hangi esikte arıza riski sıçrıyor?

Yontem:
1. Tum arizalar icin GUNLUK_SEFER_SAYISI dagilimi
2. Kantil bantlarina ayır (Q1, Q2, Q3, Q4, Q5 / 0-5%, 5-25%, 25-75%, 75-95%, 95-100%)
3. Her bantta ciddi arıza orani hesaplayıp egriyi cizmesi


In [2]:
# BOLUM 2: Sefer Sayisi Dagilimi + Bantlar
# Veriye gore karar verecegiz, sartlamadan

valid = df['GUNLUK_SEFER_SAYISI'].dropna()
print(f'Toplam gecerli kayit: {len(valid):,}')
print(f'\nSefer sayisi yuzdelikleri:')
for q in [1, 5, 25, 50, 75, 95, 99]:
    print(f'  P{q}: {valid.quantile(q/100):.1f}')

# Kantil bandı
bins = [-1, 1, 3, 6, 9, 12, 99]
labels = ['0-1','2-3','4-6','7-9','10-12','13+']
df['sefer_bant'] = pd.cut(df['GUNLUK_SEFER_SAYISI'], bins=bins, labels=labels)

bant_analiz = df.groupby('sefer_bant', observed=True).agg(
    n_ariza=('ciddi_ariza','count'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_ciddiyet=('ciddiyet_skoru','mean'),
).round(3)
print(f'\n=== SEFER BANDI × ARIZA ORANI ===')
print(bant_analiz.to_string())

# Kirilma noktasi: bantlar arasi en buyuk sicrama
print(f'\n=== KIRILMA NOKTASI (Bant gecisleri) ===')
oranlar = bant_analiz['ciddi_oran'].values
labels_l = list(bant_analiz.index)
for i in range(1, len(oranlar)):
    delta = oranlar[i] - oranlar[i-1]
    print(f'  {labels_l[i-1]} -> {labels_l[i]}: ciddi_oran {oranlar[i-1]:.3f} -> {oranlar[i]:.3f} ({delta:+.3f})')

# Chi-square: bantlar arasi fark
from scipy.stats import chi2_contingency
ct = pd.crosstab(df['sefer_bant'], df['ciddi_ariza'])
chi2, p_chi, dof, _ = chi2_contingency(ct)
print(f'\nChi-square: chi2={chi2:.2f}, p={p_chi:.6e}, dof={dof}')


Toplam gecerli kayit: 58,559

Sefer sayisi yuzdelikleri:
  P1: 1.0
  P5: 3.0
  P25: 6.0
  P50: 8.0
  P75: 11.0
  P95: 16.0
  P99: 21.0

=== SEFER BANDI × ARIZA ORANI ===
            n_ariza  ciddi_oran  ort_ciddiyet
sefer_bant                                   
0-1            1111       0.802         5.360
2-3            4126       0.712         4.732
4-6           12384       0.530         4.074
7-9           16679       0.384         3.596
10-12         14438       0.251         3.179
13+            9821       0.187         3.013

=== KIRILMA NOKTASI (Bant gecisleri) ===
  0-1 -> 2-3: ciddi_oran 0.802 -> 0.712 (-0.090)
  2-3 -> 4-6: ciddi_oran 0.712 -> 0.530 (-0.182)
  4-6 -> 7-9: ciddi_oran 0.530 -> 0.384 (-0.146)
  7-9 -> 10-12: ciddi_oran 0.384 -> 0.251 (-0.133)
  10-12 -> 13+: ciddi_oran 0.251 -> 0.187 (-0.064)

Chi-square: chi2=6519.47, p=0.000000e+00, dof=5


---
## 3. ROC Egrisi - Sefer Sayisi Esiginin Tahmin Gucu

Soru: Sefer sayisi tek basina ciddi arızayı tahmin edebilir mi? AUC nedir?

Yontem: GUNLUK_SEFER_SAYISI -> ciddi_ariza icin ROC + AUC


In [3]:
# BOLUM 3: ROC Egrisi
from sklearn.metrics import roc_auc_score, roc_curve

y = df['ciddi_ariza'].values
x = df['GUNLUK_SEFER_SAYISI'].fillna(df['GUNLUK_SEFER_SAYISI'].median()).values

auc = roc_auc_score(y, x)
fpr, tpr, thresholds = roc_curve(y, x)

print(f'=== SEFER SAYISI ROC ===')
print(f'AUC: {auc:.4f}')
if auc < 0.55:
    print('  Cok zayif tahmin gucu (tek basina kullanim icin yetersiz)')
elif auc < 0.65:
    print('  Orta zayif - tek feature olarak sinirli')
elif auc < 0.75:
    print('  Orta - feature olarak degerli')
else:
    print('  Guclu - tek feature olarak bile anlamli')

# Youden J ile en iyi esik
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
print(f'\nEn iyi esik (Youden J): {thresholds[best_idx]:.1f}')
print(f'  Bu esikte TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f}')

# Esik bazli ozet
for esik in [3, 5, 7, 10]:
    pred = (x >= esik).astype(int)
    tp = ((pred==1) & (y==1)).sum()
    fp = ((pred==1) & (y==0)).sum()
    fn = ((pred==0) & (y==1)).sum()
    tn = ((pred==0) & (y==0)).sum()
    tpr_e = tp / (tp+fn) if tp+fn else 0
    fpr_e = fp / (fp+tn) if fp+tn else 0
    print(f'  Esik >= {esik}: TPR={tpr_e:.3f}, FPR={fpr_e:.3f}')


=== SEFER SAYISI ROC ===
AUC: 0.3054
  Cok zayif tahmin gucu (tek basina kullanim icin yetersiz)

En iyi esik (Youden J): inf
  Bu esikte TPR=0.000, FPR=0.000
  Esik >= 3: TPR=0.904, FPR=0.982
  Esik >= 5: TPR=0.740, FPR=0.924
  Esik >= 7: TPR=0.533, FPR=0.801
  Esik >= 10: TPR=0.245, FPR=0.518


---
## 4. Kumulatif Yorgunluk Feature'lari (Son 7/14/30 Gun)

Soru: Yorgunluk sadece "o gun" mu, yoksa kumulatif mi?

Feature uretimi:
- son_7gun_sefer_top: Arıza tarihinden onceki 7 gun toplam sefer
- son_14gun_sefer_top: Aynisi 14 gun
- son_30gun_sefer_top: Aynisi 30 gun
- son_7gun_sefer_ort: 7 gun ortalama
- son_30gun_aktif_gun: Son 30 gunde kac gun aktif

Bu feature'lari her ariza icin hesaplayacagiz.


In [4]:
# BOLUM 4: Kumulatif Yorgunluk Feature'lari
# Her ariza icin son N gun pencereli sefer toplami

# Performans icin: ariza_model'den (KAPINO, OLAYTARIHI) listesi
ariza_keys = df[['KAPINO','OLAYTARIHI']].copy()
ariza_keys['ARIZA_TARIH'] = ariza_keys['OLAYTARIHI'].dt.normalize()

# arac_gun_sefer ile merge: her ariza icin gecmis pencereye bak
ags = arac_gun_sefer.copy()
ags['TARIH'] = pd.to_datetime(ags['TARIH'])
ags = ags.sort_values(['KAPINO','TARIH'])

# Hizli yontem: arac bazinda gun gun sefer dictionary
def compute_window_features(ariza_keys, ags, windows=[7, 14, 30]):
    result = []
    ags_by_arac = {k: g[['TARIH','GUNLUK_SEFER']].values for k, g in ags.groupby('KAPINO')}
    for idx, row in ariza_keys.iterrows():
        kapi = row['KAPINO']
        tarih = row['ARIZA_TARIH']
        if kapi not in ags_by_arac:
            result.append({'idx': idx} | {f'son_{w}g_top': np.nan for w in windows} | {f'son_{w}g_ort': np.nan for w in windows} | {f'son_{w}g_aktif': np.nan for w in windows})
            continue
        arr = ags_by_arac[kapi]
        feats = {'idx': idx}
        for w in windows:
            start = tarih - pd.Timedelta(days=w)
            mask = (arr[:,0] >= start) & (arr[:,0] < tarih)
            window_seferleri = arr[mask, 1]
            feats[f'son_{w}g_top'] = window_seferleri.sum() if len(window_seferleri) else 0
            feats[f'son_{w}g_ort'] = window_seferleri.mean() if len(window_seferleri) else 0
            feats[f'son_{w}g_aktif'] = len(window_seferleri)
        result.append(feats)
    return pd.DataFrame(result).set_index('idx')

print('Kumulatif feature hesaplaniyor (~2-3 dakika)...')
window_feats = compute_window_features(ariza_keys, ags)
df = df.merge(window_feats, left_index=True, right_index=True, how='left')

print(f'\n=== KUMULATIF FEATURE ISTATISTIK ===')
for col in ['son_7g_top','son_14g_top','son_30g_top','son_30g_aktif']:
    s = df[col].dropna()
    print(f'{col:25s}  mean={s.mean():.1f}, median={s.median():.1f}, P95={s.quantile(0.95):.1f}')


Kumulatif feature hesaplaniyor (~2-3 dakika)...

=== KUMULATIF FEATURE ISTATISTIK ===
son_7g_top                 mean=51.8, median=52.0, P95=86.0
son_14g_top                mean=100.9, median=103.0, P95=161.0
son_30g_top                mean=206.0, median=213.0, P95=326.0
son_30g_aktif              mean=21.8, median=24.0, P95=29.0


---
## 5. Korelasyon: Yorgunluk Feature'lari vs Ciddi Ariza

Soru: Hangi pencereli feature en guclu sinyal veriyor?

Yontem: Pearson, Spearman, Pointbiserial.


In [5]:
# BOLUM 5: Korelasyon + Bant Analizi
from scipy.stats import pearsonr, spearmanr, pointbiserialr

features = ['GUNLUK_SEFER_SAYISI','son_7g_top','son_14g_top','son_30g_top',
            'son_7g_ort','son_30g_aktif']

print('=== FEATURE × CIDDIYET KORELASYON ===')
print(f'{"Feature":25s} {"Pearson_r":>10s} {"p":>10s} {"Spearman":>10s} {"Ciddi_r":>10s}')
print('-'*70)
for f in features:
    s = df[f].dropna()
    y_aligned = df.loc[s.index, 'ciddiyet_skoru']
    c_aligned = df.loc[s.index, 'ciddi_ariza']
    pr, pp = pearsonr(s, y_aligned)
    sr, sp = spearmanr(s, y_aligned)
    cr, cp = pointbiserialr(c_aligned, s)
    print(f'{f:25s} {pr:+10.4f} {pp:>10.4f} {sr:+10.4f} {cr:+10.4f}')

# ANOVA: en guclu feature'i bantla
en_guclu = 'son_30g_top'
print(f'\n=== ANOVA: {en_guclu} bant analizi ===')
df['yorgun_bant'] = pd.qcut(df[en_guclu], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
bant_y = df.groupby('yorgun_bant', observed=True).agg(
    n=('ciddi_ariza','count'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_skor=('ciddiyet_skoru','mean'),
).round(3)
print(bant_y.to_string())

from scipy.stats import f_oneway
gruplar = [g['ciddiyet_skoru'].values for _, g in df.groupby('yorgun_bant', observed=True)]
f_stat, p_f = f_oneway(*gruplar)
print(f'\nANOVA: F={f_stat:.2f}, p={p_f:.6e}')


=== FEATURE × CIDDIYET KORELASYON ===
Feature                    Pearson_r          p   Spearman    Ciddi_r
----------------------------------------------------------------------
GUNLUK_SEFER_SAYISI          -0.2668     0.0000    -0.2791    -0.3097
son_7g_top                   -0.0315     0.0000    -0.0335    -0.0440
son_14g_top                  -0.0271     0.0000    -0.0302    -0.0388
son_30g_top                  -0.0144     0.0005    -0.0174    -0.0237
son_7g_ort                   -0.0316     0.0000    -0.0381    -0.0512
son_30g_aktif                -0.0084     0.0411    -0.0075    -0.0028

=== ANOVA: son_30g_top bant analizi ===
                 n  ciddi_oran  ort_skor
yorgun_bant                             
Q1           11826       0.399     3.648
Q2           11791       0.392     3.645
Q3           11842       0.377     3.605
Q4           11422       0.364     3.569
Q5           11678       0.368     3.582

ANOVA: F=4.35, p=1.615085e-03


---
## 6. Random Null Testi - Yorgunluk Etkisi Gercek mi?

Soru: Sefer-ciddi_ariza iliskisi sansa bagli mi?

Yontem: ciddi_ariza degerlerini rastgele perturbe et, korelasyonun bu kadar yuksek olma olasiligi.


In [6]:
# BOLUM 6: Random Null
np.random.seed(42)
y = df['ciddi_ariza'].values
x = df['son_30g_top'].fillna(df['son_30g_top'].median()).values
gercek_r = np.corrcoef(x, y)[0,1]

print(f'Gercek r (son_30g_top × ciddi_ariza): {gercek_r:.4f}')

rastgele = []
for i in range(1000):
    y_shuf = np.random.permutation(y)
    r = np.corrcoef(x, y_shuf)[0,1]
    rastgele.append(r)
rastgele = np.array(rastgele)

print(f'Random null ortalama: {rastgele.mean():.4f}')
print(f'Random null %95 CI: [{np.percentile(rastgele,2.5):.4f}, {np.percentile(rastgele,97.5):.4f}]')
print(f'Gercek r > rastgele max: {gercek_r > rastgele.max()}')
print(f'p-value (one-sided): {(rastgele >= abs(gercek_r)).mean():.4f}')

if gercek_r > rastgele.max():
    print('\n=> YORGUNLUK ETKISI GERCEK (rastgele degil)')
else:
    print('\n=> RASTGELE DURUM ICINDE - dikkatli ol')


Gercek r (son_30g_top × ciddi_ariza): -0.0237
Random null ortalama: 0.0000
Random null %95 CI: [-0.0074, 0.0080]
Gercek r > rastgele max: False
p-value (one-sided): 0.0000

=> RASTGELE DURUM ICINDE - dikkatli ol


---
## 7. Confounder Kontrolu - Yas + Garaj + Hat Egimi

Soru: Yorgunluk etkisinin ne kadari saf, ne kadari yastan/garajdan geliyor?

Yontem: Multiple regression M1->M4


In [7]:
# BOLUM 7: Confounder Kontrolu
import statsmodels.api as sm
from statsmodels.formula.api import ols
import json as _json

# Hat egimi (Analiz 3'ten)
with open(r'panel_data\hat_elevation.json', encoding='utf-8') as f:
    hat_elev_raw = _json.load(f)
hat_elev = pd.DataFrame([
    {'HATKODU': k, 'rakim_fark': v.get('rakım_farkı', 0), 'tirmanma_m': v.get('tırmanma_m', 0)}
    for k, v in hat_elev_raw.items()
])
def mm_norm(s, q=None):
    if q is not None:
        s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)
hat_elev['norm_rakim'] = mm_norm(hat_elev['rakim_fark'], q=0.99)
hat_elev['norm_tirm'] = mm_norm(hat_elev['tirmanma_m'], q=0.99)
hat_elev['egim_puan'] = (hat_elev['norm_rakim'] * 0.4 + hat_elev['norm_tirm'] * 0.6).round(1)

# Arac bazinda ortalama egim
ah = sefer.merge(hat_elev[['HATKODU','egim_puan']], on='HATKODU', how='left')
arac_egim = ah.dropna(subset=['egim_puan']).groupby('KAPINO').apply(
    lambda g: np.average(g['egim_puan'], weights=g[SEFER_KOL].clip(lower=0.01))
).reset_index(name='ort_egim')

# Arac bazinda ariza ozet
arac_yorgun = df.groupby('KAPINO').agg(
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi_oran=('ciddi_ariza','mean'),
    son_30g_mean=('son_30g_top','mean'),
    son_7g_mean=('son_7g_top','mean'),
).reset_index()

arac_full_y = arac_yorgun.merge(arac_meta[['KAPINO','GARAJ','yas']], on='KAPINO').merge(
    arac_egim, on='KAPINO', how='left').merge(
    arac_istat[['KAPINO','sefer_top']], on='KAPINO', how='left')
arac_full_y['log_sefer'] = np.log1p(arac_full_y['sefer_top'].fillna(0))
arac_full_y = arac_full_y.dropna(subset=['yas','log_sefer','ort_egim','son_30g_mean','GARAJ'])
print(f'Regresyon seti: {len(arac_full_y):,} arac')

# Modeller
m1 = ols('ort_skor ~ son_30g_mean', data=arac_full_y).fit()
m2 = ols('ort_skor ~ son_30g_mean + yas', data=arac_full_y).fit()
m3 = ols('ort_skor ~ son_30g_mean + yas + log_sefer', data=arac_full_y).fit()
m4 = ols('ort_skor ~ son_30g_mean + yas + log_sefer + ort_egim + C(GARAJ)', data=arac_full_y).fit()

print()
print('=== R^2 KARSILASTIRMA ===')
print(f'M1 (sadece yorgunluk):      R^2={m1.rsquared:.4f}, k_yorgun={m1.params.get("son_30g_mean", 0):+.5f}')
print(f'M2 (+ yas):                 R^2={m2.rsquared:.4f}, k_yorgun={m2.params.get("son_30g_mean", 0):+.5f}')
print(f'M3 (+ log_sefer):           R^2={m3.rsquared:.4f}, k_yorgun={m3.params.get("son_30g_mean", 0):+.5f}')
print(f'M4 (+ egim + garaj):        R^2={m4.rsquared:.4f}, k_yorgun={m4.params.get("son_30g_mean", 0):+.5f}')

k1 = m1.params.get('son_30g_mean', 0)
k4 = m4.params.get('son_30g_mean', 0)
if abs(k4) < abs(k1) * 0.3:
    print('\nYORGUNLUK katsayisi M4`te BUYUK OLCUDE DUSTU -> confounder etkisi buyuk')
elif abs(k4) < abs(k1) * 0.7:
    print('\nYORGUNLUK katsayisi M4`te bir miktar dustu -> kismi confounder')
else:
    print('\nYORGUNLUK katsayisi M4`te KORUNDU -> bagimsiz etki')


Regresyon seti: 3,509 arac

=== R^2 KARSILASTIRMA ===
M1 (sadece yorgunluk):      R^2=0.0112, k_yorgun=-0.00146
M2 (+ yas):                 R^2=0.0506, k_yorgun=-0.00113
M3 (+ log_sefer):           R^2=0.0699, k_yorgun=+0.00137
M4 (+ egim + garaj):        R^2=0.2571, k_yorgun=+0.00088

YORGUNLUK katsayisi M4`te bir miktar dustu -> kismi confounder


C:\Users\asus\AppData\Local\Temp\ipykernel_6408\462548701.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  arac_egim = ah.dropna(subset=['egim_puan']).groupby('KAPINO').apply(


---
## 8. Sofor vs Arac Yorgunlugu Ayrimi

Soru: Sefer yorgunlugu sofor sebepli mi yoksa arac sebepli mi?

Yontem: Ayni arac × farkli sofor ile farkli arac × ayni sofor pencerelerini karsilastir.


In [8]:
# BOLUM 8: Sofor vs Arac Yorgunlugu
# Sofor bazinda agirligi olculmus arıza orani
df_v = df.dropna(subset=['SOFOR_SICILNO']).copy()
df_v['SOFOR_SICILNO'] = df_v['SOFOR_SICILNO'].astype(str)

# Arac bazinda farkli sofor sayisi
arac_sofor_sayi = df_v.groupby('KAPINO')['SOFOR_SICILNO'].nunique().reset_index(name='farkli_sofor')
sofor_arac_sayi = df_v.groupby('SOFOR_SICILNO')['KAPINO'].nunique().reset_index(name='farkli_arac')

print(f'=== Arac × Sofor Cesitliligi ===')
print(f'Arac basina farkli sofor: median={arac_sofor_sayi["farkli_sofor"].median()}, P95={arac_sofor_sayi["farkli_sofor"].quantile(0.95)}')
print(f'Sofor basina farkli arac: median={sofor_arac_sayi["farkli_arac"].median()}, P95={sofor_arac_sayi["farkli_arac"].quantile(0.95)}')

# Yorgunluk × ciddi_ariza: arac vs sofor bazli
df_v['son_30g_top_band'] = pd.qcut(df_v['son_30g_top'], q=3, labels=['dusuk','orta','yuksek'], duplicates='drop')

print(f'\n=== Arac yorgunlugu × ciddi_ariza (sofor sabit tutmadan) ===')
print(df_v.groupby('son_30g_top_band', observed=True)['ciddi_ariza'].agg(['mean','count']).round(3))

# Aynı sofor degisik araç
print(f'\n=== Yuksek yorgunluk arızalarında en cok kullanilan vardiya saatleri ===')
yuksek = df_v[df_v['son_30g_top_band']=='yuksek']
if 'SAAT' in df_v.columns:
    print(yuksek['SAAT'].value_counts().head(5))


=== Arac × Sofor Cesitliligi ===
Arac basina farkli sofor: median=12.0, P95=31.0
Sofor basina farkli arac: median=7.0, P95=20.0

=== Arac yorgunlugu × ciddi_ariza (sofor sabit tutmadan) ===
                   mean  count
son_30g_top_band              
dusuk             0.398  19740
orta              0.376  19510
yuksek            0.366  19309

=== Yuksek yorgunluk arızalarında en cok kullanilan vardiya saatleri ===
SAAT
16    1441
15    1436
8     1394
7     1386
17    1320
Name: count, dtype: int64


---
## 9. ML Feature Turetimi + Bant Analizi

Soru: Hangi feature'lar ML modeline guvenle eklenebilir?


In [9]:
# BOLUM 9: ML Feature Bant Analizi
candidates = ['GUNLUK_SEFER_SAYISI','son_7g_top','son_14g_top','son_30g_top','son_7g_ort','son_30g_aktif']
results = []
for f in candidates:
    s = df[f].dropna()
    y = df.loc[s.index, 'ciddiyet_skoru']
    c = df.loc[s.index, 'ciddi_ariza']
    pr, pp = stats.pearsonr(s, y)
    sr, _ = stats.spearmanr(s, y)
    cr, _ = stats.pointbiserialr(c, s)
    # Bant analizi
    try:
        bant = pd.qcut(s, q=4, labels=False, duplicates='drop')
        gruplar = [y[bant==b].values for b in range(bant.max()+1)]
        f_stat, p_f = stats.f_oneway(*gruplar)
    except Exception:
        f_stat, p_f = np.nan, np.nan
    results.append({'feature': f, 'pearson': pr, 'p': pp, 'spearman': sr, 'ciddi_r': cr, 'ANOVA_F': f_stat, 'ANOVA_p': p_f})

res_df = pd.DataFrame(results).round(4)
print(res_df.to_string(index=False))

print('\n=== GUCLU FEATURE LISTESI (|r|>0.05 ve p<0.05) ===')
for _, row in res_df.iterrows():
    if abs(row['pearson']) > 0.05 and row['p'] < 0.05:
        print(f'  {row["feature"]:25s}  r={row["pearson"]:+.4f}')


            feature  pearson      p  spearman  ciddi_r   ANOVA_F  ANOVA_p
GUNLUK_SEFER_SAYISI  -0.2668 0.0000   -0.2791  -0.3097 1511.4046   0.0000
         son_7g_top  -0.0315 0.0000   -0.0335  -0.0440   21.2815   0.0000
        son_14g_top  -0.0271 0.0000   -0.0302  -0.0388   15.8804   0.0000
        son_30g_top  -0.0144 0.0005   -0.0174  -0.0237    5.4274   0.0010
         son_7g_ort  -0.0316 0.0000   -0.0381  -0.0512   18.8621   0.0000
      son_30g_aktif  -0.0084 0.0411   -0.0075  -0.0028    4.8486   0.0023

=== GUCLU FEATURE LISTESI (|r|>0.05 ve p<0.05) ===
  GUNLUK_SEFER_SAYISI        r=-0.2668


---
## 10. Time-Based Leakage Testi

Soru: Yorgunluk feature'lari zamansal split'te stabil mi?

Yontem: Veri 6 ay -> ilk 3 ay (train) + son 3 ay (test). Feature'lar train'de hesaplanip test'e uygulansin.


In [10]:
# BOLUM 10: Time-Based Leakage
df_sorted = df.sort_values('OLAYTARIHI').reset_index(drop=True)
n = len(df_sorted)
train = df_sorted.iloc[:n//2].copy()
test = df_sorted.iloc[n//2:].copy()
print(f'Train: {len(train):,} ariza ({train["OLAYTARIHI"].min()} -> {train["OLAYTARIHI"].max()})')
print(f'Test:  {len(test):,} ariza ({test["OLAYTARIHI"].min()} -> {test["OLAYTARIHI"].max()})')

# Her feature icin: train'de hesaplanmis arac bazli ort'u test'e tasimak
for f in ['GUNLUK_SEFER_SAYISI','son_30g_top']:
    # Train'de arac × ortalama feature
    train_arac = train.groupby('KAPINO')[f].mean().to_dict()
    test_arac_feat = test['KAPINO'].map(train_arac)
    valid = test_arac_feat.notna()

    full_r = np.corrcoef(df[f].fillna(0), df['ciddiyet_skoru'])[0,1]
    if valid.sum() > 100:
        time_r = np.corrcoef(test_arac_feat[valid], test.loc[valid,'ciddiyet_skoru'])[0,1]
        dusus = (1 - abs(time_r)/abs(full_r)) * 100 if abs(full_r) > 0.01 else 0
        print(f'\n{f}:')
        print(f'  Full-data r = {full_r:+.4f}')
        print(f'  Time-aware r = {time_r:+.4f}')
        print(f'  Dusus: %{dusus:.1f}')
        if dusus > 50:
            print(f'  LEAKAGE RISKI! ATLA')
        elif dusus > 20:
            print(f'  Orta risk - dikkat')
        else:
            print(f'  Stabil - guvenli')


Train: 29,279 ariza (2025-01-01 00:30:56 -> 2025-04-07 17:16:17.798000)
Test:  29,280 ariza (2025-04-07 17:19:26.929000 -> 2025-06-30 23:29:56)

GUNLUK_SEFER_SAYISI:
  Full-data r = -0.2668
  Time-aware r = -0.0193
  Dusus: %92.8
  LEAKAGE RISKI! ATLA

son_30g_top:
  Full-data r = -0.0144
  Time-aware r = -0.0063
  Dusus: %56.3
  LEAKAGE RISKI! ATLA


---
## 11. Vaka Analizi - En Yorgun Aracların Profili

Soru: En yorgun araclar kim? Yas + garaj + sofor cesitliligi?


In [11]:
# BOLUM 11: Vaka Analizi
en_yorgun = arac_full_y.nlargest(20, 'son_30g_mean')[['KAPINO','GARAJ','yas','son_30g_mean','ort_skor','ciddi_oran']]
print('=== EN YORGUN 20 ARAC ===')
print(en_yorgun.to_string(index=False))

print('\n=== EN DINLENMIS 20 ARAC ===')
en_dinlenmis = arac_full_y.nsmallest(20, 'son_30g_mean')[['KAPINO','GARAJ','yas','son_30g_mean','ort_skor','ciddi_oran']]
print(en_dinlenmis.to_string(index=False))

print('\n=== GARAJ × YORGUNLUK ORTALAMASI ===')
gy = arac_full_y.groupby('GARAJ').agg(
    n=('KAPINO','count'),
    yas_ort=('yas','mean'),
    son_30g_mean=('son_30g_mean','mean'),
    ort_skor=('ort_skor','mean'),
).round(2).sort_values('son_30g_mean', ascending=False)
print(gy.to_string())


=== EN YORGUN 20 ARAC ===
KAPINO     GARAJ  yas  son_30g_mean  ort_skor  ciddi_oran
 M5733   Anadolu 19.0    486.437500  4.282500    0.437500
 M2933   Anadolu 19.0    474.375000  2.891250    0.187500
 M5731   Anadolu 19.0    452.653846  4.035000    0.384615
 O4157  Sarıgazi 12.0    452.500000  3.618333    0.333333
 E9339   Topkapı  1.0    431.500000  3.520000    1.000000
 K2798 Kağıthane 12.0    422.800000  3.872000    0.400000
 O2310   KURTKÖY 12.0    419.583333  3.768333    0.458333
 O6504     Yunus 12.0    400.571429  5.000000    0.571429
 K2893 Kağıthane 12.0    395.250000  4.695000    0.375000
 O3367     Yunus 12.0    390.916667  2.689167    0.166667
 O5959     Yunus 12.0    390.100000  4.185000    0.500000
 K6025 Kağıthane 12.0    389.714286  3.214286    0.142857
 M4020   Anadolu 18.0    389.578947  3.375789    0.368421
 K6039 Kağıthane 12.0    385.857143  3.801429    0.428571
 K3438 Kağıthane 12.0    382.600000  2.624000    0.200000
 M5752   Anadolu 19.0    380.437500  3.605000 

---
## 12. Operasyonel Kirilma Esigi - Karar Mekanizmasi

Soru: Hangi sefer esigi operasyonel uyari icin kullanilabilir?


In [12]:
# BOLUM 12: Operasyonel Esik
print('=== OPERASYONEL ESIK ONERILERI ===')
print('Esik > arıza riski %50 artiyor mu?')
genel_ciddi = df['ciddi_ariza'].mean()
print(f'Genel ciddi_ariza orani: {genel_ciddi:.4f}\n')

for esik in [3, 5, 7, 10, 12, 15]:
    alt = df[df['GUNLUK_SEFER_SAYISI'] < esik]['ciddi_ariza'].mean()
    ust = df[df['GUNLUK_SEFER_SAYISI'] >= esik]['ciddi_ariza'].mean()
    if alt > 0:
        lift = ust / alt
        print(f'Esik >= {esik}: alt_oran={alt:.4f}, ust_oran={ust:.4f}, lift={lift:.2f}x')

# Son 30 gun esikleri
print('\n=== SON 30 GUN TOPLAM SEFER ESIKLERI ===')
genel_skor = df['ciddiyet_skoru'].mean()
for esik in [50, 100, 200, 300, 500]:
    s_alt = df[df['son_30g_top'] < esik]['ciddiyet_skoru'].mean()
    s_ust = df[df['son_30g_top'] >= esik]['ciddiyet_skoru'].mean()
    if pd.notna(s_alt) and pd.notna(s_ust):
        print(f'son_30g_top >= {esik}: alt_skor={s_alt:.3f}, ust_skor={s_ust:.3f}, fark={s_ust-s_alt:+.3f}')


=== OPERASYONEL ESIK ONERILERI ===
Esik > arıza riski %50 artiyor mu?
Genel ciddi_ariza orani: 0.3801

Esik >= 3: alt_oran=0.7637, ust_oran=0.3610, lift=0.47x
Esik >= 5: alt_oran=0.6764, ust_oran=0.3294, lift=0.49x
Esik >= 7: alt_oran=0.5897, ust_oran=0.2899, lift=0.49x
Esik >= 10: alt_oran=0.4898, ust_oran=0.2251, lift=0.46x
Esik >= 12: alt_oran=0.4370, ust_oran=0.1970, lift=0.45x
Esik >= 15: alt_oran=0.3993, ust_oran=0.1718, lift=0.43x

=== SON 30 GUN TOPLAM SEFER ESIKLERI ===
son_30g_top >= 50: alt_skor=3.687, ust_skor=3.606, fark=-0.082
son_30g_top >= 100: alt_skor=3.656, ust_skor=3.604, fark=-0.052
son_30g_top >= 200: alt_skor=3.640, ust_skor=3.587, fark=-0.054
son_30g_top >= 300: alt_skor=3.608, ust_skor=3.623, fark=+0.015
son_30g_top >= 500: alt_skor=3.610, ust_skor=3.711, fark=+0.102


---
## 13. Sonuc Ozeti ve Karar

Bu bolum SONUCLAR.md'ye girecek bulgulari konsolide eder.


In [13]:
# BOLUM 13: Konsolide Ozet
print('=' * 60)
print('ANALIZ 6 KONSOLIDE OZET')
print('=' * 60)
print()
print('1. SEFER SAYISI ARIZAYI TAHMIN EDIYOR MU?')
print(f'   AUC = {auc:.4f}')
print()
print('2. KIRILMA NOKTASI VAR MI?')
print('   Bant gecisleri yukarida raporlandi')
print()
print('3. KUMULATIF YORGUNLUK GUCLU FEATURE MI?')
for _, row in res_df.iterrows():
    if abs(row['pearson']) > 0.05 and row['p'] < 0.05:
        print(f'   + {row["feature"]} (r={row["pearson"]:+.4f})')
print()
print('4. CONFOUNDER ALTINDA YORGUNLUK ETKISI KORUNUYOR MU?')
print(f'   M1 k_yorgun={m1.params.get("son_30g_mean", 0):+.5f}')
print(f'   M4 k_yorgun={m4.params.get("son_30g_mean", 0):+.5f}')
print()
print('5. LEAKAGE TESTI - Yukarida raporlandi')
print()
print('=> Detayli yorumlar SONUCLAR.md dosyasinda')


ANALIZ 6 KONSOLIDE OZET

1. SEFER SAYISI ARIZAYI TAHMIN EDIYOR MU?
   AUC = 0.3054

2. KIRILMA NOKTASI VAR MI?
   Bant gecisleri yukarida raporlandi

3. KUMULATIF YORGUNLUK GUCLU FEATURE MI?
   + GUNLUK_SEFER_SAYISI (r=-0.2668)

4. CONFOUNDER ALTINDA YORGUNLUK ETKISI KORUNUYOR MU?
   M1 k_yorgun=-0.00146
   M4 k_yorgun=+0.00088

5. LEAKAGE TESTI - Yukarida raporlandi

=> Detayli yorumlar SONUCLAR.md dosyasinda


---
## 14. GUZERGAHUZUNLUK Kumulatif (KM Bazli Yorgunluk)

Soru: Sefer sayisi degil ama **toplam mesafe** (km) yorgunluk sinyali veriyor mu?

Mantik: 5 km'lik 10 sefer != 30 km'lik 10 sefer. Mesafe daha temiz fiziksel yipranma proxy'si.

Yontem:
1. sefer_temiz'den arac × gun bazinda GUZERGAHUZUNLUK toplami (km olarak)
2. Her ariza icin son 7/14/30 gun kumulatif km
3. Korelasyon + bant analizi + leakage testi

Veri notu: TOPLAMKM karisik (bazi outlier odometre okumasi) -> GUZERGAHUZUNLUK kullaniyoruz (temiz).


In [14]:
# BOLUM 14: GUZERGAHUZUNLUK Kumulatif
# sefer_temiz buyuk (10M+ satir), sadece gerekli kolonlari yukle

print('sefer_temiz yukleniyor (sadece KM kolonlari)...')
sefer_km = pd.read_csv('../panel_data/temiz_veri/sefer_temiz.csv',
                       usecols=['KAPINO','BASLANGICZAMANI','GUZERGAHUZUNLUK','GERCEKLESENGUZERGAHUZUNLUK'],
                       low_memory=False)
print(f'Sefer kayit: {len(sefer_km):,}, arac: {sefer_km["KAPINO"].nunique():,}')

sefer_km['TARIH'] = pd.to_datetime(sefer_km['BASLANGICZAMANI'], format='mixed', errors='coerce').dt.normalize()
n_before = len(sefer_km)
sefer_km = sefer_km.dropna(subset=['TARIH'])
print(f'Gecersiz tarih filtresi: {n_before:,} -> {len(sefer_km):,} ({n_before-len(sefer_km):,} kayit dusuruldu)')
# GERCEKLESEN tercih edilir, NaN ise GUZERGAH
sefer_km['km'] = sefer_km['GERCEKLESENGUZERGAHUZUNLUK'].fillna(sefer_km['GUZERGAHUZUNLUK']) / 1000  # metre -> km

# Outlier filtresi: bir sefer > 200 km mantiksiz (Istanbul ici)
sefer_km = sefer_km[(sefer_km['km'] > 0) & (sefer_km['km'] < 200)]
print(f'Outlier filtresi sonrasi: {len(sefer_km):,} kayit')
print(f'KM dagilimi (sefer basina): median={sefer_km["km"].median():.1f}, P95={sefer_km["km"].quantile(0.95):.1f}, P99={sefer_km["km"].quantile(0.99):.1f}')

# Arac × gun bazli toplam km
arac_gun_km = sefer_km.groupby(['KAPINO','TARIH'])['km'].sum().reset_index()
arac_gun_km.columns = ['KAPINO','TARIH','GUNLUK_KM']
print(f'Arac × gun: {len(arac_gun_km):,} kayit')
print(f'Gunluk km dagilimi: median={arac_gun_km["GUNLUK_KM"].median():.1f}, P95={arac_gun_km["GUNLUK_KM"].quantile(0.95):.1f}')

# Her ariza icin son N gun kumulatif km
agk_by_arac = {k: g[['TARIH','GUNLUK_KM']].values for k, g in arac_gun_km.groupby('KAPINO')}

def compute_km_windows(ariza_keys, agk_by_arac, windows=[7, 14, 30]):
    result = []
    for idx, row in ariza_keys.iterrows():
        kapi = row['KAPINO']
        tarih = row['ARIZA_TARIH']
        feats = {'idx': idx}
        if kapi not in agk_by_arac:
            for w in windows:
                feats[f'son_{w}g_km'] = np.nan
            result.append(feats)
            continue
        arr = agk_by_arac[kapi]
        for w in windows:
            start = tarih - pd.Timedelta(days=w)
            mask = (arr[:,0] >= start) & (arr[:,0] < tarih)
            feats[f'son_{w}g_km'] = arr[mask, 1].sum() if mask.sum() else 0
        result.append(feats)
    return pd.DataFrame(result).set_index('idx')

print('\nKumulatif km feature hesaplaniyor (~2-3 dakika)...')
km_feats = compute_km_windows(ariza_keys, agk_by_arac)
df = df.merge(km_feats, left_index=True, right_index=True, how='left')
print('Tamamlandi.')

print('\n=== KUMULATIF KM ISTATISTIK ===')
for col in ['son_7g_km','son_14g_km','son_30g_km']:
    s = df[col].dropna()
    print(f'{col:15s}  mean={s.mean():.1f}, median={s.median():.1f}, P95={s.quantile(0.95):.1f}, P99={s.quantile(0.99):.1f}')

# Korelasyon
print('\n=== KORELASYON ===')
from scipy.stats import pearsonr, spearmanr, pointbiserialr
for f in ['son_7g_km','son_14g_km','son_30g_km']:
    s = df[f].dropna()
    y = df.loc[s.index, 'ciddiyet_skoru']
    c = df.loc[s.index, 'ciddi_ariza']
    pr, pp = pearsonr(s, y)
    sr, _ = spearmanr(s, y)
    cr, _ = pointbiserialr(c, s)
    print(f'{f:15s}  Pearson r={pr:+.4f} p={pp:.4f}  Spearman={sr:+.4f}  Ciddi_r={cr:+.4f}')

# Bant analizi (en guclu km feature'i)
print('\n=== BANT ANALIZI: son_30g_km ===')
df['km_bant'] = pd.qcut(df['son_30g_km'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
bant_km = df.groupby('km_bant', observed=True).agg(
    n=('ciddi_ariza','count'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_skor=('ciddiyet_skoru','mean'),
).round(3)
print(bant_km.to_string())

from scipy.stats import f_oneway
gruplar = [g['ciddiyet_skoru'].values for _, g in df.groupby('km_bant', observed=True)]
f_stat, p_f = f_oneway(*gruplar)
print(f'\nANOVA: F={f_stat:.2f}, p={p_f:.6e}')

# Leakage testi
print('\n=== LEAKAGE TESTI (son_30g_km) ===')
df_sort = df.sort_values('OLAYTARIHI').reset_index(drop=True)
n_half = len(df_sort) // 2
train_km = df_sort.iloc[:n_half]
test_km = df_sort.iloc[n_half:]
train_arac_km = train_km.groupby('KAPINO')['son_30g_km'].mean().to_dict()
test_arac_feat = test_km['KAPINO'].map(train_arac_km)
valid = test_arac_feat.notna()
full_r = pearsonr(df['son_30g_km'].fillna(0), df['ciddiyet_skoru'])[0]
if valid.sum() > 100:
    time_r = pearsonr(test_arac_feat[valid].values, test_km.loc[valid,'ciddiyet_skoru'].values)[0]
    dusus = (1 - abs(time_r)/abs(full_r)) * 100 if abs(full_r) > 0.01 else 0
    print(f'  Full r={full_r:+.4f}, Time-aware r={time_r:+.4f}, Dusus %{dusus:.1f}')
    if dusus > 50:
        print('  LEAKAGE RISKI')
    else:
        print('  Stabil/orta')


sefer_temiz yukleniyor (sadece KM kolonlari)...
Sefer kayit: 10,162,086, arac: 6,761
Gecersiz tarih filtresi: 10,162,086 -> 8,395,872 (1,766,214 kayit dusuruldu)
Outlier filtresi sonrasi: 8,388,418 kayit
KM dagilimi (sefer basina): median=20.8, P95=50.3, P99=71.6
Arac × gun: 998,283 kayit
Gunluk km dagilimi: median=183.5, P95=395.3

Kumulatif km feature hesaplaniyor (~2-3 dakika)...
Tamamlandi.

=== KUMULATIF KM ISTATISTIK ===
son_7g_km        mean=1135.8, median=994.7, P95=2586.2, P99=3162.9
son_14g_km       mean=2216.4, median=1953.8, P95=4874.2, P99=5848.3
son_30g_km       mean=4534.4, median=4024.7, P95=9816.5, P99=11784.6

=== KORELASYON ===
son_7g_km        Pearson r=-0.0085 p=0.0389  Spearman=-0.0173  Ciddi_r=-0.0039
son_14g_km       Pearson r=-0.0069 p=0.0955  Spearman=-0.0155  Ciddi_r=-0.0006
son_30g_km       Pearson r=+0.0030 p=0.4627  Spearman=-0.0031  Ciddi_r=+0.0112

=== BANT ANALIZI: son_30g_km ===
             n  ciddi_oran  ort_skor
km_bant                             


---
## 15. Bitis Gecikmesi - Programa Uyum (Schedule Delay)

Soru: Aracin programlanan bitis saatini ne kadar asti? Bu hem trafik hem operasyonel kaymayi yakalar.

Onceki metrik (rota suresi orani) sadece "rota icindeki trafik"i olcuyordu, basta geciken seferleri kaciriyordu. Bu sefer:

Bitis_gecikmesi (dk) = BITISZAMANI - TAHMINIBITISZAMANI

Pozitif = gec bitti (gecikme), negatif = erken bitti, sifir = zamaninda.

Yontem:
1. sefer_temiz'den BITISZAMANI - TAHMINIBITISZAMANI hesapla (dakika)
2. Outlier filtresi: -60 dk ile +180 dk arasi (3 saatten fazla gecikme veri hatasi)
3. Arac × gun ortalama bitis gecikmesi
4. Her ariza icin son 7/14/30 gun ortalama
5. Korelasyon + bant + leakage


In [15]:
# Defensive: bagimliliklari garanti et
import pandas as pd
import numpy as np
if "df" not in globals() or "ariza_keys" not in globals():
    raise RuntimeError("Bu hucre Bolum 1-4 sonrasi calistirilmali (df + ariza_keys gerekli).")

# BOLUM 15: Bitis Gecikmesi (Schedule Delay)
print("sefer_temiz icin bitis gecikmesi hesaplaniyor...")
sefer_plan = pd.read_csv("../panel_data/temiz_veri/sefer_temiz.csv",
                         usecols=["KAPINO","BASLANGICZAMANI","BITISZAMANI","TAHMINIBITISZAMANI"],
                         low_memory=False)

# Tarih parse (gecersizler NaT)
sefer_plan["baslangic_dt"] = pd.to_datetime(sefer_plan["BASLANGICZAMANI"], format="mixed", errors="coerce")
sefer_plan["bitis_dt"] = pd.to_datetime(sefer_plan["BITISZAMANI"], format="mixed", errors="coerce")
sefer_plan["tahmin_bitis_dt"] = pd.to_datetime(sefer_plan["TAHMINIBITISZAMANI"], format="mixed", errors="coerce")
n_before = len(sefer_plan)
sefer_plan = sefer_plan.dropna(subset=["baslangic_dt","bitis_dt","tahmin_bitis_dt"])
print(f"Gecersiz tarih filtresi: {n_before:,} -> {len(sefer_plan):,}")

sefer_plan["TARIH"] = sefer_plan["baslangic_dt"].dt.normalize()
# Bitis gecikmesi (dakika cinsinden)
sefer_plan["bitis_gecikme"] = (sefer_plan["bitis_dt"] - sefer_plan["tahmin_bitis_dt"]).dt.total_seconds() / 60

print()
print("=== HAM BITIS GECIKMESI DAGILIMI (dakika) ===")
print(sefer_plan["bitis_gecikme"].describe().round(2).to_string())

# Outlier filtresi: -60 dk ile +180 dk arasi
n_before = len(sefer_plan)
sefer_plan = sefer_plan[(sefer_plan["bitis_gecikme"] > -60) & (sefer_plan["bitis_gecikme"] < 180)]
print(f"\nOutlier filtresi (-60 dk, +180 dk): {n_before:,} -> {len(sefer_plan):,}")
print(f"Bitis gecikmesi (filtreli): median={sefer_plan['bitis_gecikme'].median():.2f} dk, P95={sefer_plan['bitis_gecikme'].quantile(0.95):.2f} dk, mean={sefer_plan['bitis_gecikme'].mean():.2f} dk")

# Arac × gun ortalama bitis gecikmesi
arac_gun_plan = sefer_plan.groupby(["KAPINO","TARIH"])["bitis_gecikme"].mean().reset_index()
arac_gun_plan.columns = ["KAPINO","TARIH","GUNLUK_GECIKME_DK"]
print(f"\nArac × gun kayit sayisi: {len(arac_gun_plan):,}")
print(f"Gunluk ortalama gecikme: median={arac_gun_plan['GUNLUK_GECIKME_DK'].median():.2f} dk, P95={arac_gun_plan['GUNLUK_GECIKME_DK'].quantile(0.95):.2f} dk")

# Her ariza icin son N gun ortalama
agp_by_arac = {k: g[["TARIH","GUNLUK_GECIKME_DK"]].values for k, g in arac_gun_plan.groupby("KAPINO")}

def compute_delay_windows(ariza_keys, agp_by_arac, windows=[7, 14, 30]):
    result = []
    for idx, row in ariza_keys.iterrows():
        kapi = row["KAPINO"]
        tarih = row["ARIZA_TARIH"]
        feats = {"idx": idx}
        if kapi not in agp_by_arac:
            for w in windows:
                feats[f"son_{w}g_gecikme_dk"] = np.nan
            result.append(feats)
            continue
        arr = agp_by_arac[kapi]
        for w in windows:
            start = tarih - pd.Timedelta(days=w)
            mask = (arr[:,0] >= start) & (arr[:,0] < tarih)
            window = arr[mask, 1]
            feats[f"son_{w}g_gecikme_dk"] = window.mean() if len(window) else np.nan
        result.append(feats)
    return pd.DataFrame(result).set_index("idx")

print("\nBitis gecikmesi feature hesaplaniyor (~1-2 dakika)...")
plan_feats = compute_delay_windows(ariza_keys, agp_by_arac)
# Onceki yanlis feature'lari dusur (varsa)
for col in ["son_7g_gecikme","son_14g_gecikme","son_30g_gecikme"]:
    if col in df.columns:
        df = df.drop(columns=[col])
df = df.merge(plan_feats, left_index=True, right_index=True, how="left")

print("\n=== BITIS GECIKMESI ISTATISTIK (arac × pencere) ===")
for col in ["son_7g_gecikme_dk","son_14g_gecikme_dk","son_30g_gecikme_dk"]:
    s = df[col].dropna()
    print(f"{col:25s}  n={len(s):,}, mean={s.mean():.2f} dk, std={s.std():.2f} dk, P95={s.quantile(0.95):.2f} dk")

# Korelasyon
print("\n=== KORELASYON (ciddi_ariza + ciddiyet_skoru) ===")
from scipy.stats import pearsonr, spearmanr, pointbiserialr
for f in ["son_7g_gecikme_dk","son_14g_gecikme_dk","son_30g_gecikme_dk"]:
    s = df[f].dropna()
    y = df.loc[s.index, "ciddiyet_skoru"]
    c = df.loc[s.index, "ciddi_ariza"]
    pr, pp = pearsonr(s, y)
    sr, _ = spearmanr(s, y)
    cr, _ = pointbiserialr(c, s)
    print(f"{f:25s}  Pearson r={pr:+.4f} p={pp:.4f}  Spearman={sr:+.4f}  Ciddi_r={cr:+.4f}")

# Bant analizi
print("\n=== BANT ANALIZI: son_30g_gecikme_dk ===")
df["gecikme_bant"] = pd.qcut(df["son_30g_gecikme_dk"], q=5, labels=["Q1_zamaninda","Q2","Q3","Q4","Q5_cok_gec"], duplicates="drop")
bant_plan = df.groupby("gecikme_bant", observed=True).agg(
    n=("ciddi_ariza","count"),
    ciddi_oran=("ciddi_ariza","mean"),
    ort_skor=("ciddiyet_skoru","mean"),
    ort_gecikme=("son_30g_gecikme_dk","mean"),
).round(3)
print(bant_plan.to_string())

from scipy.stats import f_oneway
gruplar = [g["ciddiyet_skoru"].values for _, g in df.groupby("gecikme_bant", observed=True)]
f_stat, p_f = f_oneway(*gruplar)
print(f"\nANOVA: F={f_stat:.2f}, p={p_f:.6e}")

# Leakage testi
print("\n=== LEAKAGE TESTI (son_30g_gecikme_dk) ===")
df_sort = df.sort_values("OLAYTARIHI").reset_index(drop=True)
n_half = len(df_sort) // 2
train_p = df_sort.iloc[:n_half]
test_p = df_sort.iloc[n_half:]
train_arac_p = train_p.groupby("KAPINO")["son_30g_gecikme_dk"].mean().to_dict()
test_arac_feat = test_p["KAPINO"].map(train_arac_p)
valid = test_arac_feat.notna()
s_full = df["son_30g_gecikme_dk"].dropna()
full_r = pearsonr(s_full, df.loc[s_full.index, "ciddiyet_skoru"])[0]
if valid.sum() > 100:
    time_r = pearsonr(test_arac_feat[valid].values, test_p.loc[valid,"ciddiyet_skoru"].values)[0]
    dusus = (1 - abs(time_r)/abs(full_r)) * 100 if abs(full_r) > 0.01 else 0
    print(f"  Full r={full_r:+.4f}, Time-aware r={time_r:+.4f}, Dusus %{dusus:.1f}")
    if dusus > 50:
        print("  LEAKAGE RISKI")
    elif dusus > 30:
        print("  Orta risk")
    else:
        print("  Stabil")

print("\n=== YORUM ===")
print("Pozitif r ve dar leakage -> trafik/program kaymasi yorgunluk sinyali")
print("Eger sinyal yine zayifsa: bu veride yorgunluk gercekten yok ya da olculemiyor")


sefer_temiz icin bitis gecikmesi hesaplaniyor...
Gecersiz tarih filtresi: 10,162,086 -> 7,733,518

=== HAM BITIS GECIKMESI DAGILIMI (dakika) ===
count     7733518.00
mean        35030.51
std        667694.14
min      -9030453.28
25%             5.92
50%            15.20
75%            28.45
max      12883763.53

Outlier filtresi (-60 dk, +180 dk): 7,733,518 -> 7,695,676
Bitis gecikmesi (filtreli): median=15.12 dk, P95=60.50 dk, mean=19.66 dk

Arac × gun kayit sayisi: 978,406
Gunluk ortalama gecikme: median=18.82 dk, P95=52.59 dk

Bitis gecikmesi feature hesaplaniyor (~1-2 dakika)...

=== BITIS GECIKMESI ISTATISTIK (arac × pencere) ===
son_7g_gecikme_dk          n=56,675, mean=21.83 dk, std=10.48 dk, P95=41.19 dk
son_14g_gecikme_dk         n=57,266, mean=21.83 dk, std=9.11 dk, P95=38.51 dk
son_30g_gecikme_dk         n=57,627, mean=21.96 dk, std=8.04 dk, P95=36.65 dk

=== KORELASYON (ciddi_ariza + ciddiyet_skoru) ===
son_7g_gecikme_dk          Pearson r=-0.0129 p=0.0022  Spearman=-0.0219